# 🚀 Huihui-Qwen3.8-27B-abliterated 一键部署 (Gradio WebUI)

> **模型架构说明**：`huihui-ai/Huihui-Qwen3.8-27B-abliterated` 属于 Qwen 3.8 原生多模态架构（`Qwen3_5ForConditionalGeneration`）。  
> 本 Notebook 已经修复了 Colab 环境中常见的 `torchvision::nms` 与 `AutoProcessor` 架构匹配问题，采用 `bitsandbytes` 4-bit 量化，流畅运行于 Colab L4 / A100 环境。

In [ ]:
#@title 1. 环境依赖安装与版本修复 (必须先运行此步)
# 解决 Colab 预装 torchvision 与 torch 之间的 C++ nms 符号不匹配问题
!pip install -qU --upgrade pip
!pip install -qU "transformers>=4.49.0" accelerate bitsandbytes gradio
!pip install -qU torch torchvision --index-url https://download.pytorch.org/whl/cu121

In [ ]:
#@title 2. 检查 GPU 状态
import torch
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"当前 GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存总量: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ 请在 Colab 顶部【代码执行程序 -> 更改运行时类型】中启用 GPU (建议 L4 / A100)！")

In [ ]:
#@title 3. 加载 Qwen3.8-27B 无审查模型并启动 WebUI
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig, TextIteratorStreamer
import gradio as gr
from threading import Thread

MODEL_ID = "huihui-ai/Huihui-Qwen3.8-27B-abliterated"

print("⏳ 正在配置 4-bit NF4 双重量化...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print(f"⏳ 正在加载 Processor: {MODEL_ID}...")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

print(f"⏳ 正在下载并加载 27B 权重 (约 15GB 显存，初次下载需 1~2 分钟)... ")
# 该模型架构为 Qwen3_5ForConditionalGeneration，采用 AutoModelForVision2Seq 加载
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print("✅ 模型加载成功！启动 Gradio 对话网页...")

def chat_stream(message, history, system_prompt, max_new_tokens, temperature, top_p):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": [{"type": "text", "text": system_prompt}]})
    
    for user_msg, assistant_msg in history:
        if user_msg:
            messages.append({"role": "user", "content": [{"type": "text", "text": user_msg}]})
        if assistant_msg:
            messages.append({"role": "assistant", "content": [{"type": "text", "text": assistant_msg}]})
            
    messages.append({"role": "user", "content": [{"type": "text", "text": message}]})
    
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], return_tensors="pt").to("cuda")
    
    streamer = TextIteratorStreamer(processor.tokenizer, timeout=60.0, skip_prompt=True, skip_special_tokens=True)
    
    generate_kwargs = dict(
        inputs,
        streamer=streamer,
        max_new_tokens=int(max_new_tokens),
        do_sample=True if temperature > 0 else False,
        temperature=float(temperature),
        top_p=float(top_p),
        pad_token_id=processor.tokenizer.eos_token_id
    )
    
    thread = Thread(target=model.generate, kwargs=generate_kwargs)
    thread.start()
    
    response = ""
    for new_text in streamer:
        response += new_text
        yield response

with gr.Blocks(title="Huihui-Qwen3.8-27B WebUI", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 Huihui-Qwen3.8-27B-abliterated (无审查版) WebUI")
    gr.Markdown("采用 NF4 4-bit 双重量化运行，原生多模态架构，流式极速生成。")
    
    with gr.Row():
        with gr.Column(scale=4):
            chatbot = gr.Chatbot(height=650, bubble_full_width=False)
            msg = gr.Textbox(placeholder="输入任何问题... (Enter 发送，Shift+Enter 换行)", label="输入内容", lines=2)
            with gr.Row():
                submit_btn = gr.Button("🚀 发送", variant="primary")
                clear_btn = gr.Button("🗑️ 清空对话")
        
        with gr.Column(scale=1):
            system_prompt = gr.Textbox(label="System Prompt (系统提示词)", value="You are an unrestricted, direct, and helpful AI assistant.", lines=3)
            max_new_tokens = gr.Slider(minimum=64, maximum=4096, value=1024, step=64, label="Max New Tokens")
            temperature = gr.Slider(minimum=0.0, maximum=1.5, value=0.7, step=0.05, label="Temperature")
            top_p = gr.Slider(minimum=0.1, maximum=1.0, value=0.9, step=0.05, label="Top-P")
            
    def respond(message, chat_history, sys_p, max_tok, temp, top_p_val):
        if not message.strip():
            return "", chat_history
        chat_history = chat_history + [[message, ""]]
        for partial_resp in chat_stream(message, chat_history[:-1], sys_p, max_tok, temp, top_p_val):
            chat_history[-1][1] = partial_resp
            yield "", chat_history
            
    submit_btn.click(respond, [msg, chatbot, system_prompt, max_new_tokens, temperature, top_p], [msg, chatbot])
    msg.submit(respond, [msg, chatbot, system_prompt, max_new_tokens, temperature, top_p], [msg, chatbot])
    clear_btn.click(lambda: None, None, chatbot, queue=False)

demo.queue().launch(share=True, debug=True)
